# Control de alto nivel del G1

Todos los snipets de codigo que se usan para la configuración del robot y la simulación estan explicados en el archivo walk_fixed.ipynb

In [2]:
from pathlib import Path
import os
import re
import sys

os.environ.setdefault('MUJOCO_GL', 'egl')

import mediapy as media
import mujoco
import numpy as np
import onnxruntime as ort


def find_workspace_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/home/tfg_ws/src')]
    for path in candidates:
        if (path / '../mjlab/src/mjlab').exists() and (path / '2025-tfg-diego-lopez/pruebas/G1').exists():
            return path
    raise RuntimeError('No se encontró la raíz del proyecto diego_fix')


WORKSPACE = find_workspace_root()
NOTEBOOK_DIR = WORKSPACE / '2025-tfg-diego-lopez/pruebas/G1'
MJLAB_SRC = WORKSPACE / 'mjlab/src'
POLICY_PATH = NOTEBOOK_DIR / '2026-01-26_10-42-02.onnx'

if str(MJLAB_SRC) not in sys.path:
    sys.path.insert(0, str(MJLAB_SRC))

assert POLICY_PATH.exists(), POLICY_PATH

In [3]:
from mjlab.asset_zoo.robots.unitree_g1.g1_constants import (
    FULL_COLLISION,
    G1_ACTION_SCALE,
    G1_ACTUATOR_4010,
    G1_ACTUATOR_5020,       # importa las diferentes constantes que se van ha usar
    G1_ACTUATOR_7520_14,
    G1_ACTUATOR_7520_22,
    G1_ACTUATOR_ANKLE,
    G1_ACTUATOR_WAIST,
    G1_XML,
    KNEES_BENT_KEYFRAME,
)

TIMESTEP = 0.005    # Frecuencia 200 Hz
DECIMATION = 4      # Ejecuta la policy cada 4 iteraciones (a 50 Hz)
OBS_DIM = 99        # Tamaño de la observación
ACTION_DIM = 29     # Tamaño de la inferencia
FREE_QPOS = 7       # Valor inicial de las posiciones de las articulaciones en el array qpos
FREE_QVEL = 6       # valor inicial de las velocidades de las articulaciones en el array vpos

ACTUATOR_CFGS = (
    G1_ACTUATOR_5020,
    G1_ACTUATOR_7520_14,
    G1_ACTUATOR_7520_22,
    G1_ACTUATOR_4010,       # lista de los actuadores
    G1_ACTUATOR_WAIST,
    G1_ACTUATOR_ANKLE,
)
ACTUATOR_GROUPS = [
    (cfg.joint_names_expr, cfg.stiffness, cfg.damping, cfg.effort_limit, cfg.armature)
    for cfg in ACTUATOR_CFGS
]   # obtiene para cada actuador los diferentes parámetros
INIT_ROOT_POS = np.array(KNEES_BENT_KEYFRAME.pos, dtype=np.float32)
INIT_JOINT_POS = KNEES_BENT_KEYFRAME.joint_pos # define los valores iniciales de las articulaciones con una pose predefinida

In [4]:
def spec_joint_names(spec: mujoco.MjSpec) -> list[str]:
    return [
        joint.name
        for joint in spec.joints
        if joint.type != mujoco.mjtJoint.mjJNT_FREE
    ]
# devuelve la lista de las articulaciones excluyendo las libres

def resolve_joint_positions(joint_names: list[str]) -> np.ndarray:
    values = np.zeros(len(joint_names), dtype=np.float32)
    for pattern, value in INIT_JOINT_POS.items():
        for i, name in enumerate(joint_names):
            if re.fullmatch(pattern, name):
                values[i] = value
    return values
# Construye un vector con la posición inicial de cada articulación

# en INIT_JOINT_POS los valores van ligados a un patron en el nombre, no al nombre exacto de la articulación
# de tal forma que si el nombre de la articulación coincide con la expresión regular .*_hip_pitch_joint
# la posición inicial será -0.312 y así con cada diferente expresión en INIT_JOINT_POS


def matching_names(patterns: tuple[str, ...], names: list[str]) -> list[str]:
    return [
        name
        for name in names
        if any(re.fullmatch(pattern, name) for pattern in patterns)
    ]
# crea una lista de las articulaciones cuyo nombre coincida con alguna de las expresiones regulares definidas en patterns

def add_position_actuator(spec, joint_name, stiffness, damping, effort_limit, armature):
    actuator = spec.add_actuator(name=joint_name, target=joint_name)
    actuator.trntype = mujoco.mjtTrn.mjTRN_JOINT        # el actuador actua directamente sobre un joint
    actuator.dyntype = mujoco.mjtDyn.mjDYN_NONE         # tiene control directo
    actuator.gaintype = mujoco.mjtGain.mjGAIN_FIXED     # es un controlador PD
    actuator.biastype = mujoco.mjtBias.mjBIAS_AFFINE
    actuator.gainprm[0] = stiffness                     # stiffness y damping contienen los valores que
    actuator.biasprm[1] = -stiffness                    # se usarán como Kp y Kd dentro de la formula que
    actuator.biasprm[2] = -damping                      # utiliza mujoco con esos diferentes valores
    actuator.inheritrange = 0.0
    actuator.ctrllimited = False
    actuator.forcelimited = True
    actuator.forcerange[:] = np.array([-effort_limit, effort_limit])    # limita el par máximo del actuador
    spec.joint(joint_name).armature = armature          # añade inercia al actuador
# crea un actuador de mujoco

def build_g1_model() -> tuple[mujoco.MjModel, mujoco.MjData]:
    spec = mujoco.MjSpec.from_file(str(G1_XML))                 # carga el xml de la constante definida en g1_constant.py
    joint_names = spec_joint_names(spec)                        # llama a la funcion que devuelve la lista de artuculaciones
    default_joint_pos = resolve_joint_positions(joint_names)    # llama a la funcion que devuelve la posición inicial de los actuadores

    FULL_COLLISION.edit_spec(spec)          # activa todas las colosiones del modelo
    spec.worldbody.add_geom(
        name='floor',
        type=mujoco.mjtGeom.mjGEOM_PLANE,   # añade el suelo
        size=[10, 10, 0.1],
        pos=[0, 0, 0],
    )

    name_to_default = dict(zip(joint_names, default_joint_pos))
    ctrl_default = []
    for patterns, stiffness, damping, effort_limit, armature in ACTUATOR_GROUPS:
        for joint_name in matching_names(patterns, joint_names):
            add_position_actuator(spec, joint_name, stiffness, damping, effort_limit, armature)
            ctrl_default.append(name_to_default[joint_name])    # guarda la posicion inicial de cada uno
    # crea cada uno de los actuadores que estan definidos en la lista retornada por matching_names

    key_qpos = np.concatenate([
        INIT_ROOT_POS,
        np.array([1, 0, 0, 0], dtype=np.float32),   # crea el estado inicial
        default_joint_pos,
    ])
    spec.add_key(name='init_state', qpos=key_qpos.tolist(), ctrl=ctrl_default)
    # guarda el estado y el control inicial

    model = spec.compile()                      # prepara el modelo para la simulación
    model.opt.timestep = TIMESTEP               # define los parametros de la simulación
    model.opt.iterations = 10
    model.opt.ls_iterations = 20
    if hasattr(model.opt, 'ccd_iterations'):
        model.opt.ccd_iterations = 500
    return model, mujoco.MjData(model)
    # creal el model y el data para la simulación de mujoco

model, data = build_g1_model()
print('Actuadores:', model.nu)
print('qpos:', model.nq, 'qvel:', model.nv)

Actuadores: 29
qpos: 36 qvel: 35


In [5]:
session = ort.InferenceSession(str(POLICY_PATH), providers=['CPUExecutionProvider']) # carga el modelo .onnx
input_name = session.get_inputs()[0].name       # obtiene los imputs
output_name = session.get_outputs()[0].name     # obtiene los outputs

print(input_name, session.get_inputs()[0].shape)    # imprime el tamaño de los vectores
print(output_name, session.get_outputs()[0].shape)  # imput y output

assert session.get_inputs()[0].shape == [1, OBS_DIM]    # comprueba que el tamaño sea igual al definido previamente
assert session.get_outputs()[0].shape == [1, ACTION_DIM]

obs [1, 99]
actions [1, 29]


In [6]:
def non_free_joint_names(model: mujoco.MjModel) -> list[str]:
    names = []
    for joint_id in range(model.njnt):
        if model.jnt_type[joint_id] != mujoco.mjtJoint.mjJNT_FREE:
            names.append(mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id))
    return names
# devuelve una lista con los articulaciones no libres

def actuator_names(model: mujoco.MjModel) -> list[str]:
    return [
        mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
        for i in range(model.nu)
    ]
# devuelve una lista de los nombres de los actuadores en orden

def action_scale_for_joint(joint_name: str) -> float:
    for pattern, scale in G1_ACTION_SCALE.items():
        if re.fullmatch(pattern, joint_name):
            return float(scale)
    raise KeyError(f'No hay escala de acción para {joint_name}')
# devuelve el factor de escala de un joint
# compara el nombre con cada uno de los patrones de G1_ACTION_SCALE para obtener los valores

joint_order = non_free_joint_names(model) # obtiene la lista de articulaciones
ctrl_order = actuator_names(model)        # obtiene la lista de actuadores
joint_index = {name: i for i, name in enumerate(joint_order)} # crea un dicionario para relacionar el nombre de las articulaciones con un Nº
action_scale = np.array([action_scale_for_joint(name) for name in joint_order], dtype=np.float32) # crea el array de escalas
default_joint_pos = model.key('init_state').qpos[FREE_QPOS:].astype(np.float32).copy() # obtiene las posiciones iniciales


def joint_targets_to_ctrl(joint_targets: np.ndarray) -> np.ndarray:
    return np.array([joint_targets[joint_index[name]] for name in ctrl_order], dtype=np.float32)
# reordena la lista del orden que usa la red al roden que necesita mujoco

print('Primeras articulaciones:', joint_order[:6])
print('Primeros actuadores:', ctrl_order[:6])

Primeras articulaciones: ['left_hip_pitch_joint', 'left_hip_roll_joint', 'left_hip_yaw_joint', 'left_knee_joint', 'left_ankle_pitch_joint', 'left_ankle_roll_joint']
Primeros actuadores: ['left_shoulder_pitch_joint', 'left_shoulder_roll_joint', 'left_shoulder_yaw_joint', 'left_elbow_joint', 'left_wrist_roll_joint', 'right_shoulder_pitch_joint']


In [7]:
def sensor_slice(model: mujoco.MjModel, sensor_name: str) -> slice:
    sensor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SENSOR, sensor_name)
    start = model.sensor_adr[sensor_id]
    return slice(start, start + model.sensor_dim[sensor_id])
# Todos los valores de los sensores de mujoco estan en un array grande, por lo que
# esta función retorna que valores corresponden a un determinado sensor

lin_vel_slice = sensor_slice(model, 'imu_lin_vel')  # obtiene las posiciones de los sensores correspondientes a la velocidad linear
ang_vel_slice = sensor_slice(model, 'imu_ang_vel')  # obtiene las posiciones de los sensores correspondientes a la velocidad angular
pelvis_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'pelvis') # obtiene el id de la pelvis


def projected_gravity(data: mujoco.MjData) -> np.ndarray:
    pelvis_xmat = data.xmat[pelvis_body_id].reshape(3, 3)
    return pelvis_xmat.T @ np.array([0.0, 0.0, -1.0], dtype=np.float32)
# obtiene el vector de gravedad del robot con respecto a la pelvis

def make_observation(data, last_action, command) -> np.ndarray:
    obs = np.concatenate([
        data.sensordata[lin_vel_slice],
        data.sensordata[ang_vel_slice],
        projected_gravity(data),
        data.qpos[FREE_QPOS:] - default_joint_pos, # la observación de la posición es la diferencia entre la posción actual y la predeterminada
        data.qvel[FREE_QVEL:],
        last_action,
        command,
    ]).astype(np.float32)
    assert obs.shape == (OBS_DIM,), obs.shape
    return obs
# crea la observación que usará la red

In [8]:
def reset_to_default():
    key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'init_state') # obtiene los datos de la poscición inicial
    mujoco.mj_resetDataKeyframe(model, data, key_id)    # añade el estado a mujoco
    data.ctrl[:] = joint_targets_to_ctrl(default_joint_pos) # reinicia los controles
    mujoco.mj_forward(model, data)  # recalcula la física interna
# reinicia el modelo a la posición inicial

def infer_action(last_action: np.ndarray, command: np.ndarray) -> np.ndarray:
    obs = make_observation(data, last_action, command)[None, :] # obtiene la observación
    return session.run([output_name], {input_name: obs})[0][0].astype(np.float32) # devuelve la inferencia


def apply_action(raw_action: np.ndarray):
    joint_targets = default_joint_pos + raw_action * action_scale   # Escala la acción para que la use mujoco
    data.ctrl[:] = joint_targets_to_ctrl(joint_targets) # reordena la lista y se la pasa a data.ctrl

## 8. Rollout

In [9]:
import threading
import time

twist_lock = threading.Lock()

finish_event = threading.Event()

### Tread de control

Realiza un patrón sencillo de moviminetos

In [10]:
twist = np.array([-0.6, 0.0, 0.0], dtype=np.float32)   # comando que se enviará al robot

def control_thread(data):
    global twist

    i = 0

    control_dt = 1/10
    next_control_time = 0.0
    prev_mode = -1

    while not finish_event.is_set():

        while data.time < next_control_time:
            time.sleep(0.005)
            if finish_event.is_set():
                return

        next_control_time += control_dt

        mode = (i // 20) % 4
        
        if prev_mode != mode:   # evita cojer el cerrojo continuamente, solo cambiando el twist cuando es necesario
            if(mode == 0):
                with twist_lock:
                    twist[:] = [0.5, 0.0, 0.0]
            elif(mode == 1):
                with twist_lock:
                    twist[:] = [0.0, 0.0, 0.0]
            elif(mode == 2):
                with twist_lock:
                    twist[:] = [-0.5, 0.0, 0.0]
            else:
                with twist_lock:
                    twist[:] = [0.0, 0.0, 0.8]
        prev_mode = mode

        i += 1
        if (i >= 80):
            i = 0

### Thread de inferencia

Calcula la salida de la red neuronal y envia los comandos al simulador

In [11]:
def inference_thread(data):
    last_action = np.zeros(ACTION_DIM, dtype=np.float32)    # crea una acción inicial vacía

    control_dt = 1/50
    next_control_time = 0.0

    while not finish_event.is_set():
        
        while data.time < next_control_time:
            time.sleep(0.0005)
            if finish_event.is_set():
                return

        next_control_time += control_dt

        with twist_lock:
            twist_cp = np.array(twist, dtype=np.float32, copy=True)
        raw_action = infer_action(last_action, twist_cp) # hace la inferencia
        apply_action(raw_action)    # ejecuta la acción
        last_action = raw_action    # guarda la acción que acaba de ejecutar para usarla como last_action

### Thread principal

Ejecuata la simulación

In [12]:
duration_s = 16.0    # duración de la simulación (s)
framerate = 60      # framerate del video

reset_to_default()  # reinicia a la posicón inicial
frames = []

start_x = float(data.qpos[0])   # obtiene la x inicial para calcular el avanze al final

finish_event.clear()

control_t = threading.Thread(target=control_thread, args=(data,))
inference_t = threading.Thread(target=inference_thread, args=(data,))
control_t.start()
inference_t.start()

start = time.perf_counter()

with mujoco.Renderer(model) as renderer:
    while data.time < duration_s:

        mujoco.mj_step(model, data)
        if len(frames) < data.time * framerate:     # graba el video
            renderer.update_scene(data)
            frames.append(renderer.render())


end = time.perf_counter()

print(f"Tiempo: {end - start:.3f} s")

finish_event.set()
control_t.join()
inference_t.join()

print(f'Avance aproximado: {data.qpos[0] - start_x:.2f} m') # calcula la distancia recorrida
media.show_video(frames, fps=framerate) # enseña el video


Tiempo: 2.702 s
Avance aproximado: -0.03 m


This browser does not support the video tag.


In [15]:
import imageio

skip = 4

gif_frams = frames[::skip]

# guardar como GIF
imageio.mimsave(
    "G1_walking.gif",
    gif_frams,
    fps=60
)

print("GIF guardado como simulacion.gif")

GIF guardado como simulacion.gif
